In [1]:
import pandas as pd
hsc_bed = pd.read_csv('/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Snapatac2/HSC/hsc_marker_celltype_dar.csv')
hsc_bed

,peak,cell_type
0,chr1:1128402-1128903,APC_cycling
1,chr1:2091940-2092441,APC_cycling
2,chr1:2124783-2125284,APC_cycling
3,chr1:2138061-2138562,APC_cycling
4,chr1:2230183-2230684,APC_cycling
...,...,...
1056041,chrY:7792371-7792872,VLMC
1056042,chrY:13226302-13226803,VLMC
1056043,chrY:13233901-13234402,VLMC
1056044,chrY:14586524-14587025,VLMC


In [3]:
import os

# 1. Set paths
output_dir = "/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Homer/HSC/bed"
enhancer_path = "/cluster2/huanglab/jiamao/Project/HumanSpinalCord/Work/P3/ATAC/Homer/HSC/enhancer.bed"

# Create output directory if missing
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 2. Parse peak column into chrom, start, end
# Regex split for chr1:817820-818321 style coords
# \W matches non-word chars (e.g. : and -)
temp_df = hsc_bed['peak'].str.split('[:|-]', expand=True)
hsc_bed['chrom'] = temp_df[0]
hsc_bed['start'] = temp_df[1]
hsc_bed['end'] = temp_df[2]

# 3. Add fixed columns
hsc_bed['score'] = 2026
hsc_bed['strand'] = 0

# 4. Per cell_type: write 6-column BED
# Column order: chrom, start, end, peak, 2026, 0
columns_6 = ['chrom', 'start', 'end', 'peak', 'score', 'strand']

for cell_type, group in hsc_bed.groupby('cell_type'):
    # Sanitize filename if needed
    safe_name = str(cell_type).replace("/", "_").replace(" ", "_")
    file_path = os.path.join(output_dir, f"{safe_name}.bed")
    
    # Tab-separated, no header/index
    group[columns_6].to_csv(file_path, sep='\t', header=False, index=False)

# 5. All peaks -> enhancer.bed (3 columns)
# Three columns: chrom, start, end
# Note: pooled peaks across types often need deduplication
enhancer_df = hsc_bed[['chrom', 'start', 'end']].drop_duplicates()
enhancer_df.to_csv(enhancer_path, sep='\t', header=False, index=False)